In [ ]:
import cv2  #Importing required libraries
import numpy as np
import matplotlib.pyplot as plt

def displayimage(title, image, isitbgr=True): #Defining a function to display input images
    plt.figure(figsize=(16, 8)) #Size of image displayed
    if len(image.shape) == 2: #If dimension of image is 2 then display grayscale image
        plt.imshow(image, cmap='gray')
    else:
        if isitbgr:  #If image is in bgr format then convert it to rgb format
            plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)) #Displaying image
        else:  #else directly display rgb image
            plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

def filtercomponents(mask, minarea=30, maxarea=2500, minaspect=0.001, maxaspect=6.0): #defining function so that only leopard spots mask is created and other noise or tree part is not removed in final image
    countoflabels , labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8) #8 connectivity in all 8 directions
    filteredmask = np.zeros_like(mask) #Creating a filtered version of mask so that only leopard dots are considered
    for i in range(1, countoflabels): #Looping through each label
        area = stats[i, cv2.CC_STAT_AREA] #Getting pixel area of each label i(each connected region)
        width, height = stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT] #getting bounding box height and width of each connected region
        aspect = width / height if height > 0 else 0 #computing aspect to know real approximate shape of label

        if minarea < area < maxarea and minaspect < aspect < maxaspect: #Keeping only leopard dots and not considering others
            filteredmask[labels == i] = 255 #Connected regions that are similar to leopard spots in size and shape will remain. Therefore marking such labels as white in new mask
    return filteredmask


def leopardspotsremove(inputimagepath, outputimagepath="final_result.jpg"): #Function to identify and remove dots of leopard
    bgrimage = cv2.imread(inputimagepath) #Reading input image
    displayimage("Input Image", bgrimage) #displaying input image

    grayimage = cv2.cvtColor(bgrimage, cv2.COLOR_BGR2GRAY)   #Converting bgr image to grayscale image which is easier to detect spots
    displayimage("Grayscale Image", grayimage, isitbgr=False) #displaying image

    masks = [] #Creating empty array of masks
    for blocksize in range(17, 2, -2): #For various block sizes, considering various masks that will remove small and big leopard spots.
        mask = cv2.adaptiveThreshold( #Creating mask to detect spots on leopard's body
        grayimage, #Grayscale version of image
        255, #Max pixel value one can get in binary image
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, #This threshold is calculated using guassian weighted sum neighbour values
        cv2.THRESH_BINARY_INV, #To invert actual binary output
        blocksize, #Neighbourhood block size which is used to find threshold
        11 #Constant that is separated from mean which helps to separated dark spots more efficiently
        )
        kernelsize = max(3, blocksize // 5)  # kernel growing according to blocksize
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernelsize,kernelsize)) #Creating elliptical kernel
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,   kernel, iterations=1) #performing morphological opening to remove small noise and break thin connectivity between spots
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,   kernel, iterations=2) #performing morphological closing to fill small black holes inside white regions
                                                                              #Performing 2 iterations for better cleaning

        displayimage(f"Mask (Block size={blocksize})", mask, isitbgr=False) #Displaying all masks
        masks.append(mask) #adding each mask to the array of masks

    finalmask = masks[0] #Creating one final mask from all these masks
    for m in masks[1:]: #looping through rest of the masks
        finalmask = cv2.bitwise_or(finalmask, m)

    displayimage("final mask ", finalmask, isitbgr=False) #Displaying final mask of for input image

    filteredmask = filtercomponents(finalmask, minarea=10, maxarea=20000) #Removing unwanted noise and tree part from mask
    displayimage("Filtered Leopard Spot Mask", filteredmask, isitbgr=False) #Mask finally which only has mask for leopard spots

    inpaintedimg = cv2.inpaint(bgrimage, filteredmask, inpaintRadius=2, flags=cv2.INPAINT_TELEA) #Filling mask points or leopard spots with sorrounding colour
    displayimage("Inpainted Image", inpaintedimg) #Displaying image

    feathermask = cv2.GaussianBlur(filteredmask.astype(np.float32), (31, 31), 0) / 255.0 #For smooth blending calculations
    feathermask = np.clip(feathermask, 0, 0.6)  #To ensure soft blending
    blurredimg = cv2.GaussianBlur(inpaintedimg, (31, 31), 0) #Applying gaussian blur to inpainted image

    finalimage = (1 - feathermask[..., None]) * inpaintedimg + feathermask[..., None] * blurredimg #Blending original inpainted image with its blurred version smoothly, so edges of removed spots look natural.
    finalimage = finalimage.astype(np.uint8) #For proper image display
    displayimage("Final output image", finalimage) #Displaying image

    cv2.imwrite(outputimagepath, finalimage) #For saving image

#Uploading input image given in assignment
leopardspotsremove("/content/Leapard.jpg", outputimagepath="improved_leopard_multi_pass.jpg")